# Imports

In [49]:
# biomart server
from biomart import BiomartServer

# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Path
import os
data_path = os.path.join('..', 'data')

# Progress bar
from tqdm import tqdm

# regex
import re

# Load Data

In [2]:
test_with_results = pd.read_pickle(os.path.join(data_path, '2_test_with_bart30.pkl'))

genes = test_with_results['query_gene'].unique()

# Search Ensembl

**Define Dataset**

In [3]:
# Server URL for Ensembl BioMart
server = BiomartServer("http://www.ensembl.org/biomart")

# Specify the dataset to use
dataset = server.datasets['hsapiens_gene_ensembl']

**Query**

In [ ]:
# Define batch size
batch_size = 128

# Define the attributes to retrieve
attributes = [
    'ensembl_gene_id',
    'uniprotswissprot'
]

# Initialize an empty list to store responses
responses = []

# Process genes in batches
for i in tqdm(range(29 * batch_size, len(genes), batch_size)):
    batch_genes = genes[i:i + batch_size].tolist()
    
    # Perform the search for the current batch
    response = dataset.search({
        'attributes': attributes,
        'filters': {
            'ensembl_gene_id': batch_genes
        }
    })
    
    # Append the full text response to the list
    responses.append(response.text)

# Create a DataFrame from the responses
responses_df = pd.DataFrame({0: responses})

100%|██████████| 2/2 [00:05<00:00,  2.51s/it]


# Match Responses

**Load Responses**

In [39]:
responses_df = pd.read_pickle('biomart_responses.pkl')

**Split Responses**

In [102]:
# split the responses into a list of lists
split_df = responses_df.apply(lambda x: re.split(r'[\t\n]+', x[0]), axis=1)

# remove empty strings from each list
split_df = split_df.apply(lambda x: [y for y in x if y != ''])

# remove consecutive ENSG genes from each list
split_df = split_df.apply(lambda x: [y for i, y in enumerate(x) if (not y.startswith('ENSG')) or (i+1 < len(x) and not x[i+1].startswith('ENSG'))])

# remove last element if it starts with 'ENSG'
split_df = split_df.apply(lambda x: x[:-1] if x[-1].startswith('ENSG') else x)

**Create DataFrame**

In [106]:
# put even indices of each list value in one column and odd indices in another column
stacked_data = np.hstack(split_df.to_numpy())
ensembl_data = stacked_data[::2]
uniprot_data = stacked_data[1::2]

# # Create a DataFrame from the stacked data
ensembl_to_uniprot = pd.DataFrame({
    'ensembl_gene_id': ensembl_data,
    'uniprot_id': uniprot_data
})

In [108]:
ensembl_to_uniprot.to_pickle('ensembl_to_uniprot.pkl')

# Save UniProt Ids to file

In [115]:
ensembl_to_uniprot = pd.read_pickle('ensembl_to_uniprot.pkl')

In [116]:
# Save uniprot ids to a file separated by newlines
with open('uniprot_ids.txt', 'w') as f:
    for uniprot_id in ensembl_to_uniprot['uniprot_id'].tolist():
        f.write(f"{uniprot_id}\n")